# 05 · Plot the selected TopK feature during Ida and Hagibis

Run this after notebooks 03 and 04 using the same Google Drive account. Choose an **A100 GPU with high system RAM**, then select **Runtime → Run all**. The GraphCast setup is the same one used in notebook 03.

This notebook produces panels **B and C** corresponding to Figure 4 of MacMillan & Ouellette. It reads the feature chosen by notebook 04 from `selected_feature.json`. In your committed course run, that is **feature 3288** from the epoch-8 checkpoint, with validation F1 0.252023 and test F1 0.380000.

Each panel shows the feature's post-TopK activation above ERA5 10 m wind speed. Wind speed is `sqrt(u10**2 + v10**2)` in m/s. Both rows use the UTC time displayed above each map. These times follow notebook 03's **last observed input / forecast initialization** convention. For a map at time `t`, GraphCast receives weather at `t−6 h` and `t` and forecasts `t+6 h`.

The ten displayed timestamps match the paper's figure. The paper's probing description leaves the initialization-versus-target timestamp convention ambiguous, so our convention is recorded in the saved metadata. Each date is an independent GraphCast run using ERA5 inputs.

The top row displays continuous SAE coefficients. Each row has one shared color scale across its five maps, with scales calculated from this trained SAE and these wind fields. The regional map extents are listed below.

[Paper, Figure 4](https://arxiv.org/html/2512.24440v1#S2.SS5)

## 1. Connect Drive and load the plotting code

The commit ID fixes the project code used by this notebook. All new data, feature maps and figures are saved to Drive. Completed snapshots can be reused after a runtime disconnect.

In [ ]:
# @title Connect Drive and load the project
import json
from google.colab import drive

import os
import subprocess
import sys
from pathlib import Path

# One immutable code version is shared by all five notebooks.
RELEASE = '2cf2b3a5e4357b0fff88ebf3bbc88c320e0e371b'
REPO = Path('/content') / ('ams691_project_' + RELEASE[:12])
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'init', '-q', str(REPO)], check=True)
head = subprocess.run(['git', '-C', str(REPO), 'rev-parse', '--verify', 'HEAD'],
                      capture_output=True, text=True)
if head.stdout.strip() != RELEASE:
    subprocess.run(['git', '-C', str(REPO), 'fetch', '-q', '--depth', '1',
                    'https://github.com/observerhan/graphcast-interpretability.git', RELEASE], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '-q', '--detach', RELEASE], check=True)
sys.path.insert(0, str(REPO / 'scripts'))
sys.path.insert(0, str(REPO / 'src'))

drive.mount('/content/drive')
DRIVE_PROJECT = Path('/content/drive/MyDrive/AMS691_GraphCast')
LABEL_RUN = DRIVE_PROJECT / 'cyclones_2019_2021'
DATA = REPO / 'data' / 'cyclone_2019_2021'


## 2. Load the saved feature selection

Use the same cohort and SAE run as notebook 04. The selected feature ID is read automatically. The `figure4_storm_cache` folder holds the new GraphCast activations and regional wind fields. Figures and encoded features are saved under this checkpoint's probe-results folder.

In [ ]:
# @title Select the completed probe run
COHORT = 'course'  # @param ["course", "all_guarded"]
SAE_RUN = 'topk_baseline'  # @param {type:"string"}
SAE_EXPERIMENT = 'monthly_2017_2018'  # @param {type:"string"}

from figure4_storms import STORMS, TIMES, saved_selection
from IPython.display import display
import pandas as pd

checkpoint = DRIVE_PROJECT / SAE_EXPERIMENT / 'topk_runs' / SAE_RUN / 'best.pt'
selection, probe_dir = saved_selection(LABEL_RUN, COHORT, checkpoint)
CACHE = LABEL_RUN / 'figure4_storm_cache'
OUT = probe_dir / 'figure4_storms'
winner = selection['validation']
print(f"Selected feature {winner['feature']} from checkpoint epoch {selection['experiment']['checkpoint_epoch']}.")
print(f"Validation F1 {winner['f1']:.6f}. Cyclone rule {winner['rule']} {winner['activation_cutoff']:.8g}.")
if winner['slope'] < 0:
    print('This probe associates lower feature activation with cyclone presence.')
rows = [dict(storm=storm['name'], initialization_utc=time,
             west=storm['extent'][0], east=storm['extent'][1],
             south=storm['extent'][2], north=storm['extent'][3])
        for storm in STORMS for time in storm['times']]
display(pd.DataFrame(rows))
print(f'{len(TIMES)} storm snapshots require at most {len(TIMES) * 40962 * 512 * 4 / 1e9:.2f} GB of new activations.')
print('Figure output folder:', OUT)

## 3. Prepare GraphCast and collect the ten snapshots

The repository script `scripts/setup_graphcast.py` prepares GraphCast. The collector imports the same layer-8 extraction functions as notebooks 01 and 03 from `scripts/ams691_extract.py`. The first prediction compiles the model. Each completed activation and wind field is saved immediately. On a rerun, the collector checks completed files and continues with the remaining times.

In [ ]:
# @title Prepare the GraphCast runtime
subprocess.run([sys.executable, '-u', str(REPO / 'scripts' / 'setup_graphcast.py')], check=True)


In [ ]:
# @title Collect GraphCast activations and ERA5 wind fields
import signal

graphcast_python = '/content/ams691_graphcast_env/bin/python'
command = [graphcast_python, '-u', str(REPO / 'scripts' / 'collect_figure4_storms.py'),
           str(LABEL_RUN), str(DATA), str(CACHE)]
env = {**os.environ,
       'PYTHONPATH': os.pathsep.join([str(REPO / 'scripts'), str(REPO / 'src'), os.environ.get('PYTHONPATH', '')]),
       'XLA_PYTHON_CLIENT_PREALLOCATE': 'false'}
process = subprocess.Popen(command, env=env, stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait():
        raise RuntimeError('Storm collection stopped. The error is printed above.')
except KeyboardInterrupt:
    process.send_signal(signal.SIGINT)
    process.wait()
    raise

## 4. Encode the saved feature and draw the maps

The GraphCast process finishes before the PyTorch SAE is loaded. The SAE uses the same normalization and TopK operation as notebook 04. The authors' containing-triangle mesh-to-grid mapping supplies the plotted feature values.

The geographic windows are 100–70°W and 15–40°N for Ida, and 120–165°E and 5–45°N for Hagibis. These fixed windows cover the storm sequences in the supplied figure.

In [ ]:
# @title Load the plotting packages and encode the selected feature
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'cartopy==0.25.0'], check=True)
from figure4_storms import encode_storms, draw_storms

metadata = encode_storms(LABEL_RUN, DATA, COHORT, checkpoint, CACHE, OUT)
print(f"Feature {metadata['selection']['feature']} is ready at all ten timestamps.")

In [ ]:
# @title Draw and display panels B and C
from IPython.display import Image, display

figure_files = draw_storms(CACHE, OUT)
for path in figure_files:
    if path.suffix == '.png':
        display(Image(filename=str(path)))
print('Saved figure files')
for path in figure_files:
    print(path)
print('Times, model identity, selected feature and color scales:', OUT / 'figure4_metadata.json')

The output folder contains `figure4b_ida.png`, `figure4b_ida.pdf`, `figure4c_hagibis.png` and `figure4c_hagibis.pdf`. The PDF files preserve coastlines and labels as vector graphics. `figure4_metadata.json` records the checkpoint, feature selection, ten times, geographic windows and color scales.

These maps provide qualitative examples of feature activation during the two storms. Notebook 04 reports the held-out F1 for the saved probe evaluation.